In [1]:
import pandas as pd
import numpy as np
import altair as alt
import json

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:

URL = "https://data.cdc.gov/resource/swc5-untb.csv?$limit=500000"
df_raw = pd.read_csv(URL)
df_raw = df_raw[df_raw["datavaluetypeid"] == "AgeAdjPrv"].copy()

/opt/conda/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (10,11) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [3]:
df = df_raw.pivot_table(
    index=["stateabbr", "statedesc", "locationname", "locationid", "totalpopulation"],
    columns="measureid",
    values="data_value"
).reset_index()
df.columns.name = None
df["totalpopulation"] = pd.to_numeric(df["totalpopulation"], errors="coerce")


In [4]:
rename = {
    "OBESITY":    "obesity_pct",
    "DIABETES":   "diabetes_pct",
    "BPHIGH":     "hypertension_pct",
    "CSMOKING":   "smoking_pct",
    "DEPRESSION": "depression_pct",
    "LPA":        "no_exercise_pct",
}
df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

health_cols = ["obesity_pct", "diabetes_pct", "hypertension_pct",
               "smoking_pct", "depression_pct", "no_exercise_pct"]
df = df.dropna(subset=health_cols)

print(f"Working dataframe: {df.shape[0]} counties × {df.shape[1]} columns")

Working dataframe: 2956 counties × 45 columns


In [5]:
from altair import topo_feature

COUNTIES_URL = "https://cdn.jsdelivr.net/npm/us-atlas@3/counties-10m.json"

# Build a lookup: FIPS (int) → health values
map_df = df[["locationid", "locationname", "statedesc"] + health_cols].copy()
map_df["locationid"] = map_df["locationid"].astype(str).str.zfill(5)
map_df = map_df.rename(columns={"locationid": "id"})

# Dropdown input
dropdown = alt.binding_select(
    options=health_cols,
    labels=[
        "Obesity (%)", "Diabetes (%)", "Hypertension (%)",
        "Smoking (%)", "Depression (%)", "Physical Inactivity (%)"
    ],
    name="Health Metric: "
)
metric_sel = alt.selection_point(
    fields=["metric"],
    bind=dropdown,
    value=[{"metric": "no_exercise_pct"}]
)

# wide format + transform_fold
map_df_wide = map_df.copy()  

background = alt.Chart(
    alt.topo_feature(COUNTIES_URL, "states")
).mark_geoshape(fill="#e8e8e8", stroke="white", strokeWidth=0.5)

choropleth = alt.Chart(
    alt.topo_feature(COUNTIES_URL, "counties")
).mark_geoshape(stroke="white", strokeWidth=0.2).transform_lookup(
    lookup="id",
    from_=alt.LookupData(
        map_df_wide, "id",
        health_cols + ["locationname", "statedesc"]
    )
).transform_fold(
    health_cols,
    as_=["metric", "value"]
).transform_filter(
    metric_sel
).encode(
    color=alt.Color(
        "value:Q",
        scale=alt.Scale(scheme="orangered", domainMin=5),
        legend=alt.Legend(title="Prevalence (%)", orient="bottom-right")
    ),
    tooltip=[
        alt.Tooltip("locationname:N", title="County"),
        alt.Tooltip("statedesc:N", title="State"),
        alt.Tooltip("value:Q", title="Prevalence (%)", format=".1f"),
    ]
).add_params(metric_sel).project("albersUsa")

chart1 = (background + choropleth).properties(
    width=750, height=460,
    title=alt.TitleParams(
        "How Healthy Is Your County? Select a Health Metric to Explore",
        fontSize=18, anchor="start"
    )
).configure_view(stroke=None)

chart1.save("chart1_map.json")
print("✓ chart1_map.json saved")

✓ chart1_map.json saved


*pic 2*

In [6]:
state_avg = df.groupby("statedesc")["obesity_pct"].mean().reset_index()
state_avg.columns = ["state", "obesity_pct"]
state_avg["obesity_pct"] = state_avg["obesity_pct"].round(1)

top10    = state_avg.nlargest(10, "obesity_pct").assign(group="Highest Obesity")
bottom10 = state_avg.nsmallest(10, "obesity_pct").assign(group="Lowest Obesity")
combined = pd.concat([top10, bottom10])

chart2 = alt.Chart(combined).mark_bar().encode(
    x=alt.X("obesity_pct:Q",
            scale=alt.Scale(domain=[0, 50]),
            title="Average Obesity Rate (%)"),
    y=alt.Y("state:N",
            sort=alt.EncodingSortField("obesity_pct", order="descending"),
            title=None),
    color=alt.Color("group:N",
                    scale=alt.Scale(domain=["Highest Obesity", "Lowest Obesity"],
                                    range=["#d73027", "#4575b4"]),
                    legend=alt.Legend(title="Group")),
    tooltip=[
        alt.Tooltip("state:N", title="State"),
        alt.Tooltip("obesity_pct:Q", title="Avg Obesity (%)", format=".1f"),
    ]
).properties(
    width=500, height=380,
    title=alt.TitleParams(
        "Which States Have the Highest and Lowest Obesity Rates?",
        fontSize=15, anchor="start"
    )
)

chart2.save("chart2_state_obesity.json")
print("✓ chart2_state_obesity.json saved")

✓ chart2_state_obesity.json saved


In [7]:
# smoking vs heart disease proxy (hypertension), no_exercise vs diabetes
scatter_df = df[["obesity_pct", "diabetes_pct", "smoking_pct",
                  "no_exercise_pct", "hypertension_pct", "statedesc"]].dropna()

# Split into 4 US regions for color
region_map = {
    "Connecticut":"Northeast","Maine":"Northeast","Massachusetts":"Northeast",
    "New Hampshire":"Northeast","Rhode Island":"Northeast","Vermont":"Northeast",
    "New Jersey":"Northeast","New York":"Northeast","Pennsylvania":"Northeast",
    "Illinois":"Midwest","Indiana":"Midwest","Michigan":"Midwest",
    "Ohio":"Midwest","Wisconsin":"Midwest","Iowa":"Midwest","Kansas":"Midwest",
    "Minnesota":"Midwest","Missouri":"Midwest","Nebraska":"Midwest",
    "North Dakota":"Midwest","South Dakota":"Midwest",
    "Delaware":"South","Florida":"South","Georgia":"South","Maryland":"South",
    "North Carolina":"South","South Carolina":"South","Virginia":"South",
    "District of Columbia":"South","West Virginia":"South","Alabama":"South",
    "Kentucky":"South","Mississippi":"South","Tennessee":"South",
    "Arkansas":"South","Louisiana":"South","Oklahoma":"South","Texas":"South",
    "Arizona":"West","Colorado":"West","Idaho":"West","Montana":"West",
    "Nevada":"West","New Mexico":"West","Utah":"West","Wyoming":"West",
    "Alaska":"West","California":"West","Hawaii":"West","Oregon":"West",
    "Washington":"West",
}
scatter_df = scatter_df.copy()
scatter_df["region"] = scatter_df["statedesc"].map(region_map).fillna("Other")

chart3 = alt.Chart(scatter_df.sample(min(2000, len(scatter_df)), random_state=42)).mark_circle(
    size=30, opacity=0.5
).encode(
    x=alt.X("no_exercise_pct:Q", title="Physical Inactivity Rate (%)"),
    y=alt.Y("diabetes_pct:Q",    title="Diabetes Rate (%)"),
    color=alt.Color("region:N",
                    scale=alt.Scale(scheme="tableau10"),
                    legend=alt.Legend(title="US Region")),
    tooltip=[
        alt.Tooltip("locationname:N", title="County") if "locationname" in scatter_df.columns
        else alt.Tooltip("statedesc:N", title="State"),
        alt.Tooltip("no_exercise_pct:Q", title="Inactivity (%)", format=".1f"),
        alt.Tooltip("diabetes_pct:Q",    title="Diabetes (%)",   format=".1f"),
        alt.Tooltip("region:N",          title="Region"),
    ]
).properties(
    width=500, height=360,
    title=alt.TitleParams(
        "Physical Inactivity vs. Diabetes Rate by US Region",
        fontSize=15, anchor="start"
    )
)

# add locationname back for tooltip if not already there
scatter_df2 = df[["locationname","obesity_pct","diabetes_pct","smoking_pct",
                   "no_exercise_pct","hypertension_pct","statedesc"]].dropna()
scatter_df2 = scatter_df2.copy()
scatter_df2["region"] = scatter_df2["statedesc"].map(region_map).fillna("Other")

chart3 = alt.Chart(scatter_df2.sample(min(2000, len(scatter_df2)), random_state=42)).mark_circle(
    size=30, opacity=0.5
).encode(
    x=alt.X("no_exercise_pct:Q", title="Physical Inactivity Rate (%)"),
    y=alt.Y("diabetes_pct:Q",    title="Diabetes Rate (%)"),
    color=alt.Color("region:N",
                    scale=alt.Scale(scheme="tableau10"),
                    legend=alt.Legend(title="US Region")),
    tooltip=[
        alt.Tooltip("locationname:N",    title="County"),
        alt.Tooltip("statedesc:N",       title="State"),
        alt.Tooltip("no_exercise_pct:Q", title="Inactivity (%)", format=".1f"),
        alt.Tooltip("diabetes_pct:Q",    title="Diabetes (%)",   format=".1f"),
        alt.Tooltip("region:N",          title="Region"),
    ]
).properties(
    width=500, height=360,
    title=alt.TitleParams(
        "Physical Inactivity vs. Diabetes Rate by US Region",
        fontSize=15, anchor="start"
    )
)

chart3.save("chart3_scatter.json")
print("✓ chart3_scatter.json saved")

✓ chart3_scatter.json saved


In [8]:
chart1.save("chart1.html")
chart2.save("chart2.html")
chart3.save("chart3.html")